# Job 6 - merge every result file and build the summary tables

No GPU and no embeddings needed: it only reads CSV files from Drive. Run it after the other jobs,
or at any time to see partial results - missing files are reported, not fatal.

**Inputs** (in `fake_profile_paper/linkedin_repro/`)

| file | written by | contents |
|---|---|---|
| `results_raw.csv` | the interrupted `LinkedIn_multitask_reproduction.ipynb` | T1 and T1b, all seeds, HGS / random / grid |
| `results_T2_seed42.csv` | job 1 | T2 seed 42, HGS + random |
| `results_T2_seeds7_123.csv` | job 2 | T2 seeds 7 and 123 |
| `results_T2_grid.csv` | job 3 | T2 seed 42, exhaustive grid |
| `results_T3.csv` | job 4 | T3, all seeds |
| `results_ablations.csv` | job 5 | tag correction off, Full Name removed |
| `results_embeddings.csv` | job 7 | ten embedding formulations x four tasks x three seeds |

**Outputs - attach these four files:** `results_all.csv`, `results_summary.csv`,
`embedding_summary.csv`, `summary.txt`.

In [ ]:
import os, numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive'); IN_COLAB = True
except ImportError:
    IN_COLAB = False
OUT_DIR = os.environ.get("LINKEDIN_OUT_DIR",
    "/content/drive/MyDrive/fake_profile_paper/linkedin_repro" if IN_COLAB else os.path.abspath("../results/run"))

MAIN_FILES = ["results_raw.csv", "results_T2_seed42.csv", "results_T2_seeds7_123.csv",
              "results_T2_grid.csv", "results_T3.csv", "results_ablations.csv"]
EMB_FILE = "results_embeddings.csv"
REPORT = []                                   # everything printed also goes to summary.txt
def say(*a):
    s = " ".join(str(x) for x in a); print(s); REPORT.append(s)

say("=== INPUT FILES ===")
parts = []
for f in MAIN_FILES:
    p = os.path.join(OUT_DIR, f)
    if os.path.exists(p):
        d = pd.read_csv(p); d["source_file"] = f; parts.append(d)
        say("  %-26s %3d rows" % (f, len(d)))
    else:
        say("  %-26s MISSING" % f)
p = os.path.join(OUT_DIR, EMB_FILE)
E = pd.read_csv(p) if os.path.exists(p) else pd.DataFrame()
say("  %-26s %s" % (EMB_FILE, "%d rows" % len(E) if len(E) else "MISSING"))

R = (pd.concat(parts, ignore_index=True) if parts else pd.DataFrame())
if len(R):
    R["variant"] = R["variant"].fillna("main")
    R = R.drop_duplicates(subset=["task", "seed", "strategy", "variant"], keep="last")
    R.to_csv(os.path.join(OUT_DIR, "results_all.csv"), index=False)
say("\nunique main runs:", len(R))

## Completeness check - what is done and what is still missing

In [ ]:
say("\n=== COMPLETENESS ===")
expected = []
for task in ["T1", "T1b", "T2", "T3"]:
    for seed in [42, 7, 123]:
        expected += [(task, seed, "HGS", "main"), (task, seed, "Random", "main")]
for task in ["T1", "T1b", "T2"]:
    expected.append((task, 42, "Grid", "main"))
for task in ["T1", "T1b", "T2"]:
    expected += [(task, 42, "HGS", "no_tag_correction"), (task, 42, "HGS", "no_full_name")]
have = set(map(tuple, R[["task", "seed", "strategy", "variant"]].values.tolist())) if len(R) else set()
missing = [e for e in expected if e not in have]
say("main study: %d of %d expected runs present" % (len(expected) - len(missing), len(expected)))
for m in missing: say("  missing:", *m)
if len(E):
    exp_e = 10 * 4 * 3
    say("embedding screen: %d of %d expected runs present" % (len(E), exp_e))

# the interrupted run printed T2 seed 42 results before it was lost; job 1 must reproduce them
say("\n=== REPRODUCIBILITY CHECK (T2 seed 42, from the interrupted run's printed output) ===")
printed = {"HGS": (0.9917, 0.9944), "Random": (0.9930, 0.9954)}
for strat, (f1, acc) in printed.items():
    row = R[(R.task == "T2") & (R.seed == 42) & (R.strategy == strat) & (R.variant == "main")] if len(R) else []
    if len(row):
        r = row.iloc[0]
        ok = abs(r.f1_pos - f1) < 5e-4 and abs(r.accuracy - acc) < 5e-4
        say("  %-6s printed F1+ %.4f acc %.4f | rerun F1+ %.4f acc %.4f -> %s"
            % (strat, f1, acc, r.f1_pos, r.accuracy, "MATCH" if ok else "DIFFERS"))
    else:
        say("  %-6s not rerun yet (job 1)" % strat)

## Main study - search strategies, grid, ablations

In [ ]:
keys = ["accuracy", "precision_pos", "recall_pos", "f1_pos", "f1_macro", "f1_w",
        "roc_auc", "pr_auc", "mcc", "n_models"]
def boot(v, B=2000, rng=np.random.default_rng(0)):
    if len(v) < 2: return ""
    b = rng.choice(v, (B, len(v))).mean(1); return "[%.4f, %.4f]" % tuple(np.percentile(b, [2.5, 97.5]))
rows = []
for (task, strat, var), g in R.groupby(["task", "strategy", "variant"]):
    row = dict(task=task, strategy=strat, variant=var, seeds=len(g))
    for k in keys:
        if k in g:
            row[k] = "%.4f +- %.4f" % (g[k].mean(), g[k].std(ddof=1) if len(g) > 1 else 0.0)
    row["f1_pos_CI"] = boot(g["f1_pos"].values)
    rows.append(row)
SUM = pd.DataFrame(rows); SUM.to_csv(os.path.join(OUT_DIR, "results_summary.csv"), index=False)
display(SUM)

say("\n=== HGS vs MATCHED-BUDGET RANDOM SEARCH (positive-class F1, paired by seed) ===")
for task in sorted(R.task.unique()):
    m = R[(R.task == task) & (R.variant == "main")]
    h = m[m.strategy == "HGS"].set_index("seed")["f1_pos"]
    r = m[m.strategy == "Random"].set_index("seed")["f1_pos"]
    d = (h - r).dropna()
    if len(d):
        say("  %-4s mean diff %+.4f | HGS better %d, tied %d, worse %d of %d seeds"
            % (task, d.mean(), (d > 0).sum(), (d == 0).sum(), (d < 0).sum(), len(d)))

say("\n=== HGS vs EXHAUSTIVE GRID, seed 42 (test F1+ and CV objective) ===")
for task in sorted(R.task.unique()):
    m = R[(R.task == task) & (R.variant == "main") & (R.seed == 42)].set_index("strategy")
    if {"HGS", "Grid"} <= set(m.index):
        say("  %-4s HGS F1+ %.4f (cv %.4f, %d models) | Grid F1+ %.4f (cv %.4f, %d models)"
            % (task, m.loc["HGS", "f1_pos"], m.loc["HGS", "cv_fitness"], m.loc["HGS", "n_models"],
               m.loc["Grid", "f1_pos"], m.loc["Grid", "cv_fitness"], m.loc["Grid", "n_models"]))

say("\n=== ABLATIONS, seed 42 (HGS, positive-class F1) ===")
for task in sorted(R.task.unique()):
    m = R[(R.task == task) & (R.seed == 42) & (R.strategy == "HGS")].set_index("variant")["f1_pos"]
    if "main" in m and len(m) > 1:
        say("  %-4s " % task + " | ".join("%s %.4f" % (v, m[v]) for v in ["main"] + [x for x in m.index if x != "main"]))

## Embedding screen (job 7) - which formulation should replace STE?

In [ ]:
NAMES = {"V0": "STE (published baseline)", "V1": "plain mean", "V2": "R1 renormalisation",
         "V3": "R2, global mean", "V4": "R2, per-section mean", "V5": "R2 on the tag direction",
         "V6": "ZCA whitening", "V7": "STE + cosine coherence", "V8": "R2 per-section + coherence",
         "V9": "STE + L1 distances"}
if len(E):
    rows = []
    for (task, v), g in E.groupby(["task", "variant"]):
        rows.append(dict(task=task, variant=v, name=NAMES.get(v, v), seeds=len(g),
                         f1_pos="%.4f +- %.4f" % (g.f1_pos.mean(), g.f1_pos.std(ddof=1) if len(g) > 1 else 0),
                         roc_auc="%.4f" % g.roc_auc.mean(), pr_auc="%.4f" % g.pr_auc.mean(),
                         mcc="%.4f" % g.mcc.mean(), accuracy="%.4f" % g.accuracy.mean()))
    ES = pd.DataFrame(rows); ES.to_csv(os.path.join(OUT_DIR, "embedding_summary.csv"), index=False)
    display(ES)

    say("\n=== EMBEDDINGS: PAIRED DIFFERENCE AGAINST V0 (STE), positive-class F1 ===")
    wins = {}
    for task in ["T1", "T1b", "T2", "T3"]:
        base = E[(E.task == task) & (E.variant == "V0")].set_index("seed")["f1_pos"]
        if not len(base): continue
        say("\n" + task)
        for v in sorted(E.variant.unique()):
            if v == "V0": continue
            d = (E[(E.task == task) & (E.variant == v)].set_index("seed")["f1_pos"] - base).dropna()
            if len(d):
                wins.setdefault(v, []).append(d.mean())
                say("  %-3s %-28s mean %+.4f | better %d, tied %d, worse %d of %d seeds"
                    % (v, NAMES.get(v, v), d.mean(), (d > 0).sum(), (d == 0).sum(), (d < 0).sum(), len(d)))
    say("\n=== EMBEDDINGS: AVERAGE GAIN OVER STE ACROSS TASKS ===")
    for v, ds in sorted(wins.items(), key=lambda kv: -np.mean(kv[1])):
        say("  %-3s %-28s mean %+.4f over %d task(s) | improves on %d"
            % (v, NAMES.get(v, v), np.mean(ds), len(ds), sum(x > 0 for x in ds)))
else:
    say("\n(job 7 not run yet - no embedding results)")

open(os.path.join(OUT_DIR, "summary.txt"), "w").write("\n".join(REPORT))
say("\nwritten: results_all.csv, results_summary.csv, embedding_summary.csv, summary.txt")